# Train BERT-base on PolyAI/banking77

Runs the full training pipeline from `src.train.train` on Colab Pro with an
A100 GPU. Produces a checkpoint in `artifacts/model/` and a metrics report in
`artifacts/reports/`.

**Expected wall time:** ~10-15 minutes on A100 (NFR-5).
**Expected metrics:** macro-F1 >= 0.90 (NFR-1), top-3 accuracy >= 0.97 (NFR-2).

## How to use

1. Runtime -> Change runtime type -> Hardware accelerator: **A100 GPU**.
2. Run all cells.
3. The last cell zips and downloads `model.zip` for use in the local API.

## 1. Verify A100 GPU is attached

In [ ]:
!nvidia-smi

## 2. Clone the repo and install dependencies

In [ ]:
# Replace with your fork/branch as needed
!git clone https://github.com/rsalehin/bert-ticket-router.git
%cd bert-ticket-router
!pip install -q -e ".[dev]"

## 3. Configure training

In [ ]:
from pathlib import Path
from src.config import Settings

settings = Settings(
    base_model_name="bert-base-uncased",
    model_checkpoint_path=Path("artifacts/model"),
    batch_size=64,
    learning_rate=2e-5,
    num_epochs=4,
    warmup_ratio=0.1,
    weight_decay=0.01,
    seed=42,
    device="auto",     # picks "cuda" on Colab A100
    max_len=64,
    top_k=3,
    model_version="bert-base-uncased@v0.1.0",
)
print(settings.model_dump())

## 4. Train

In [ ]:
from src.train import train

metrics = train(settings=settings)
print("Final test-set metrics:")
for k in ("accuracy", "macro_f1", "top1_accuracy", "top3_accuracy", "top5_accuracy"):
    print(f"  {k}: {metrics[k]:.4f}")

## 5. Spot-check NFR thresholds

In [ ]:
assert metrics["macro_f1"] >= 0.90, f"NFR-1 missed: macro_f1={metrics['macro_f1']:.4f}"
assert metrics["top3_accuracy"] >= 0.97, f"NFR-2 missed: top3_acc={metrics['top3_accuracy']:.4f}"
print("All NFR thresholds met.")

## 6. Zip and download the checkpoint

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("model", "zip", "artifacts/model")
files.download("model.zip")

## Local installation of the trained model

After download, on your local machine:

```powershell
cd bert-ticket-router
Expand-Archive -Path model.zip -DestinationPath artifacts/model -Force
```

The FastAPI server (T-021 onward) will pick up the checkpoint at
`artifacts/model/` automatically.